In [ ]:
install.packages(c("googledrive", "caret", "glmnet", "xgboost", "tidyverse", "forecast", "corrplot"))
library(googledrive)
library(caret)
library(glmnet)
library(xgboost)
library(tidyverse)
library(forecast)
library(corrplot)

drive_auth()
drive_download("train.csv", path = "train.csv", overwrite = TRUE)
drive_download("test.csv", path = "test.csv", overwrite = TRUE)

train <- read.csv("train.csv")
test <- read.csv("test.csv")

impute_missing <- function(df, columns) {
  for (col in columns) {
    df[[col]][is.na(df[[col]])] <- median(df[[col]], na.rm = TRUE)
  }
  return(df)
}

columns_to_impute <- c("log_gdp_per_capita", "social_support", "life_expectancy",
                       "freedom_choices", "generosity", "corruption")
train <- impute_missing(train, columns_to_impute)
test <- impute_missing(test, columns_to_impute)


train <- train %>%
  mutate(
    gdp_social = log_gdp_per_capita * social_support,
    gdp_squared = log_gdp_per_capita^2,
    social_squared = social_support^2
  )w
test <- test %>%
  mutate(
    gdp_social = log_gdp_per_capita * social_support,
    gdp_squared = log_gdp_per_capita^2,
    social_squared = social_support^2
  )

x_train <- as.matrix(train %>% select(log_gdp_per_capita, social_support, life_expectancy,
                                      freedom_choices, corruption, gdp_social,
                                      gdp_squared, social_squared))
y_train <- train$happiness

x_test <- as.matrix(test %>% select(log_gdp_per_capita, social_support, life_expectancy,
                                     freedom_choices, corruption, gdp_social,
                                     gdp_squared, social_squared))

#Elastic Net Regression
set.seed(123)
alpha_values <- seq(0.1, 0.9, by = 0.1)
lambda_values <- 10^seq(-2, 3, by = 0.1)
best_alpha <- NULL
best_lambda <- NULL
best_model <- NULL
best_rmse <- Inf

for (alpha in alpha_values) {
  enet_model <- cv.glmnet(x_train, y_train, alpha = alpha, lambda = lambda_values)
  if (min(enet_model$cvm) < best_rmse) {
    best_rmse <- min(enet_model$cvm)
    best_model <- enet_model
    best_alpha <- alpha
    best_lambda <- enet_model$lambda.min
  }
}

elastic_predictions <- predict(best_model, s = best_lambda, newx = x_test)

#Time Series (ARIMA)
time_series_data <- train %>%
  group_by(year) %>%
  summarise(avg_happiness = mean(happiness, na.rm = TRUE))
ts_happiness <- ts(time_series_data$avg_happiness, start = min(time_series_data$year),
                   end = max(time_series_data$year), frequency = 1)
arima_model <- auto.arima(ts_happiness)
forecast_years <- unique(test$year)
arima_forecast <- forecast(arima_model, h = length(forecast_years))
arima_predictions <- rep(NA, nrow(test))
for (i in seq_along(forecast_years)) {
  arima_predictions[test$year == forecast_years[i]] <- arima_forecast$mean[i]
}

#XGBoost
dtrain <- xgb.DMatrix(data = x_train, label = y_train)
dtest <- xgb.DMatrix(data = x_test)
xgb_params <- list(
  objective = "reg:squarederror",
  eta = 0.1,
  max_depth = 6,
  subsample = 0.8,
  colsample_bytree = 0.8
)
xgb_model <- xgb.train(params = xgb_params, data = dtrain, nrounds = 500, verbose = 0)
xgb_predictions <- predict(xgb_model, dtest)

#Model Blending
final_predictions <- 0.6 * elastic_predictions + 0.4 * xgb_predictions

submission <- data.frame(ID = test$ID, happiness = final_predictions)
write.csv(submission, "final_blended_submission.csv", row.names = FALSE)